<a href="https://colab.research.google.com/github/toddbalwinski/ds2002-midterm-project/blob/main/ds2002_midterm_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import requests
from pathlib import Path

DATA_DIR = Path("/content/data") if Path("/content/data/walmart_transactions.csv").exists() else Path("data") if Path("data/walmart_transactions.csv").exists() else Path(".")

transactions = pd.read_csv(DATA_DIR / "walmart_transactions.csv")
stores = pd.read_csv(DATA_DIR / "store_locations.csv")
catalog_path = DATA_DIR / "product_catalog.csv"
if not catalog_path.exists():
    catalog_path = DATA_DIR / "product_catalog(1).csv"
products = pd.read_csv(catalog_path)
distribution = pd.read_csv(DATA_DIR / "distribution_centers.csv")

conn = sqlite3.connect(DATA_DIR / "inventory_and_sales.db")

daily_sales = pd.read_sql_query(
    "SELECT * FROM daily_sales_summary", conn
)

inventory = pd.read_sql_query(
    "SELECT * FROM inventory_levels", conn
)

datasets = {
    "Transactions": transactions,
    "Stores": stores,
    "Products": products,
    "Distribution Centers": distribution,
    "Daily Sales": daily_sales,
    "Inventory": inventory
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")


Transactions: (25562, 8)
Stores: (14, 8)
Products: (45, 7)
Distribution Centers: (5, 8)
Daily Sales: (3600, 5)
Inventory: (4320, 6)


In [16]:
clean_transactions = transactions.copy()

print("Original rows:", len(clean_transactions))
print("Duplicate rows:", clean_transactions.duplicated().sum())

clean_transactions = clean_transactions.drop_duplicates().copy()

print("Rows after cleaning:", len(clean_transactions))


Original rows: 25562
Duplicate rows: 30
Rows after cleaning: 25532


In [17]:
print("Unique store IDs before:",
      clean_transactions["store_id"].nunique())

clean_transactions["store_id"] = (
    clean_transactions["store_id"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace("_", "-", regex=False)
    .str.replace(r"^FL-?(\d+)$", r"FL-\1", regex=True)
)

print("Unique store IDs after:",
      clean_transactions["store_id"].nunique())

print(sorted(clean_transactions["store_id"].unique()))


Unique store IDs before: 60
Unique store IDs after: 12
['FL-078', 'FL-105', 'FL-156', 'FL-239', 'FL-287', 'FL-302', 'FL-330', 'FL-412', 'FL-445', 'FL-501', 'FL-619', 'FL-710']


In [18]:
print("Missing timestamps before:",
      clean_transactions["timestamp"].isna().sum())

clean_transactions["timestamp"] = pd.to_datetime(
    clean_transactions["timestamp"],
    format="mixed",
    errors="coerce"
)

clean_transactions["date"] = (
    clean_transactions["timestamp"].dt.normalize()
)

print("Missing timestamps after:",
      clean_transactions["timestamp"].isna().sum())

print("Earliest date:", clean_transactions["date"].min())
print("Latest date:", clean_transactions["date"].max())


Missing timestamps before: 884
Missing timestamps after: 884
Earliest date: 2024-09-01 00:00:00
Latest date: 2024-09-30 00:00:00


In [19]:
clean_transactions["quantity"] = pd.to_numeric(
    clean_transactions["quantity"],
    errors="coerce"
)

clean_transactions["unit_price"] = (
    clean_transactions["unit_price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.strip()
)

clean_transactions["unit_price"] = pd.to_numeric(
    clean_transactions["unit_price"],
    errors="coerce"
)

clean_transactions["quantity_issue"] = (
    clean_transactions["quantity"].isna()
    | (clean_transactions["quantity"] <= 0)
)

clean_transactions["price_issue"] = (
    clean_transactions["unit_price"].isna()
    | (clean_transactions["unit_price"] < 0)
)

print("Quantity issues:",
      clean_transactions["quantity_issue"].sum())
print("Price issues:",
      clean_transactions["price_issue"].sum())

clean_transactions["revenue"] = (
    clean_transactions["quantity"]
    * clean_transactions["unit_price"]
)


Quantity issues: 130
Price issues: 0


In [20]:
clean_transactions["category"] = (
    clean_transactions["category"]
    .astype(str)
    .str.strip()
    .str.title()
)

category_map = {
    "Breakfast Foods": "Breakfast",
    "Emergency Supplies": "Emergency",
    "Alcoholic Beverages": "Alcohol",
    "Games & Entertainment": "Entertainment",
    "Groceries": "Grocery",
    "Household Essentials": "Household",
    "Health & Wellness": "Health",
    "Baby Products": "Baby"
}

clean_transactions["category"] = (
    clean_transactions["category"]
    .replace(category_map)
)

print(sorted(clean_transactions["category"].unique()))
print("Number of categories:",
      clean_transactions["category"].nunique())


['Alcohol', 'Baby', 'Breakfast', 'Electronics', 'Emergency', 'Entertainment', 'Grocery', 'Health', 'Household', 'Snacks']
Number of categories: 10


In [21]:
clean_products = products.copy()

clean_products["category"] = (
    clean_products["category"]
    .str.strip()
    .str.title()
    .replace(category_map)
)

clean_products["sku"] = (
    clean_products["sku"].str.strip().str.upper()
)

print(clean_products["category"].value_counts())


category
Emergency        12
Grocery           6
Entertainment     5
Breakfast         4
Electronics       4
Baby              4
Household         4
Alcohol           2
Snacks            2
Health            2
Name: count, dtype: int64


In [22]:
sku_groups = {
    "Strawberry Pop-Tarts": [
        "PT-12", "SKU#459812",
        "POPTART-STRAW", "PT_STRAWBERRY"
    ],
    "Bottled Water": [
        "WTR-24", "WATER24PK", "WTR_24PK"
    ],
    "AA Batteries": [
        "BATT-100", "BAT-AA4", "BATT_AA_4PK"
    ],
    "Flashlights": [
        "FLASH-01", "LED-FLASH"
    ],
    "Light Beer": [
        "BEER-24", "LTBEER24"
    ],
    "Monopoly Board Game": [
        "BG-MONO", "MONOPOLY-01", "GAME-MONO"
    ],
    "Playing Cards": [
        "CARDS-01", "PLAYCARD-STD"
    ],
    "Streaming Device": [
        "STREAM-01", "ROKU-STD"
    ],
    "Canned Soup": [
        "SOUP-CAN", "CNSOUP-01"
    ],
    "White Bread": [
        "BREAD-WH", "WHITEBREAD"
    ],
    "Peanut Butter": [
        "PB-16", "PNUTBTR-16"
    ],
    "Emergency Candles": [
        "CNDL-EM", "CANDLE-EMRG"
    ],
    "First Aid Kit": [
        "FAID-01", "FIRSTAID-KIT"
    ],
    "Portable Charger": [
        "CHRG-PORT", "PORTCHARGER"
    ],
    "Chips Variety Pack": [
        "CHIPS-VP", "CHIPVARIETY"
    ],
    "Paper Towels": [
        "PTOWEL-6", "PAPERTOWEL6"
    ],
    "Toilet Paper": [
        "TP-12", "TOILETPAPER12"
    ],
    "Diapers": [
        "DIAP-S3", "DIAPERS-3"
    ],
    "Baby Formula": [
        "BFORM-01", "BABYFORMULA"
    ],
    "Heavy Duty Tarp": [
        "TARP-HD", "HDTARP-01"
    ]
}

sku_map = {
    sku: family
    for family, skus in sku_groups.items()
    for sku in skus
}

clean_transactions["sku"] = (
    clean_transactions["sku"]
    .astype(str)
    .str.strip()
    .str.upper()
)

clean_transactions["product_family"] = (
    clean_transactions["sku"].map(sku_map)
)

print("Unmapped SKUs:",
      clean_transactions["product_family"].isna().sum())

print(
    clean_transactions.loc[
        clean_transactions["product_family"]
        == "Strawberry Pop-Tarts",
        ["sku", "product_family"]
    ].drop_duplicates()
)


Unmapped SKUs: 0
              sku        product_family
21  PT_STRAWBERRY  Strawberry Pop-Tarts
27     SKU#459812  Strawberry Pop-Tarts
47          PT-12  Strawberry Pop-Tarts
52  POPTART-STRAW  Strawberry Pop-Tarts


In [23]:
clean_stores = stores.copy()

clean_stores["store_id"] = (
    clean_stores["store_id"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace("_", "-", regex=False)
    .str.replace(r"^FL-?(\d+)$", r"FL-\1", regex=True)
)

clean_stores = clean_stores.drop_duplicates(
    subset=["store_id"]
).copy()

print("Unique stores:", len(clean_stores))
print("Missing coordinates:")
print(clean_stores[["latitude", "longitude"]].isna().sum())


Unique stores: 12
Missing coordinates:
latitude     1
longitude    1
dtype: int64


In [24]:
for df in [daily_sales, inventory]:
    df["store_id"] = (
        df["store_id"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace("_", "-", regex=False)
        .str.replace(r"^FL-?(\d+)$", r"FL-\1", regex=True)
    )

daily_sales["category"] = (
    daily_sales["category"]
    .astype(str)
    .str.strip()
    .str.title()
    .replace(category_map)
)

daily_sales["date"] = pd.to_datetime(
    daily_sales["date"], errors="coerce"
)

inventory["date"] = pd.to_datetime(
    inventory["date"], errors="coerce"
)

inventory["sku"] = (
    inventory["sku"].astype(str).str.strip().str.upper()
)

inventory["product_family"] = inventory["sku"].map(sku_map)


In [25]:
clean_transactions = clean_transactions.merge(
    clean_stores[
        ["store_id", "city", "region", "latitude", "longitude"]
    ],
    on="store_id",
    how="left",
    validate="many_to_one"
)


print("Transactions without store match:",
      clean_transactions["city"].isna().sum())


analysis_sales = clean_transactions.loc[
    clean_transactions["date"].notna()
    & clean_transactions["quantity"].notna()
    & clean_transactions["product_family"].notna()
].copy()

analysis_sales["gross_units"] = (
    analysis_sales["quantity"].clip(lower=0)
)

analysis_sales["returned_units"] = (
    -analysis_sales["quantity"].clip(upper=0)
)

analysis_sales["net_units"] = (
    analysis_sales["quantity"]
)

print("Analysis-ready transactions:", len(analysis_sales))
print("Gross units:", analysis_sales["gross_units"].sum())
print("Returned units:", analysis_sales["returned_units"].sum())
print("Net units:", analysis_sales["net_units"].sum())



Transactions without store match: 0
Analysis-ready transactions: 24648
Gross units: 47897
Returned units: 237
Net units: 47660


In [29]:
print("Cleaning validation")

print("Original transactions:", len(transactions))
print("Deduplicated transactions:", len(clean_transactions))
print("Analysis-ready transactions:", len(analysis_sales))

print("Missing dates:",
      clean_transactions["date"].isna().sum())

print("Potential return transactions:",
      (clean_transactions["quantity"] < 0).sum())

print("Zero-quantity transactions:",
      (clean_transactions["quantity"] == 0).sum())

print("Missing product mappings:",
      clean_transactions["product_family"].isna().sum())

print("Missing store matches:",
      clean_transactions["city"].isna().sum())

print("Product families:",
      clean_transactions["product_family"].nunique())

print("Store IDs:",
      clean_transactions["store_id"].nunique())


Cleaning validation
Original transactions: 25562
Deduplicated transactions: 25532
Analysis-ready transactions: 24648
Missing dates: 884
Potential return transactions: 130
Zero-quantity transactions: 0
Missing product mappings: 0
Missing store matches: 0
Product families: 20
Store IDs: 12


In [27]:
import time

WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"
RETRYABLE = {429, 500, 502, 503, 504}

def fetch_daily_weather(start_date="2024-09-01", end_date="2024-09-30",
                        latitude=28.54, longitude=-81.38, attempts=3):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "precipitation_sum,wind_speed_10m_max,pressure_msl_mean",
        "timezone": "America/New_York"
    }
    for attempt in range(attempts):
        try:
            response = requests.get(WEATHER_URL, params=params, timeout=25)
            if response.status_code in RETRYABLE and attempt < attempts - 1:
                time.sleep(2 ** attempt)
                continue
            response.raise_for_status()
            payload = response.json()
            if "daily" not in payload:
                raise ValueError(f"Weather response missing daily data: {str(payload)[:300]}")
            result = pd.DataFrame(payload["daily"]).rename(columns={"time": "date"})
            result["date"] = pd.to_datetime(result["date"], errors="raise").dt.normalize()
            result["source"] = "Open-Meteo historical archive"
            result["timezone"] = payload.get("timezone", "America/New_York")
            if result["date"].duplicated().any():
                raise ValueError("Weather response has repeated dates")
            return result
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(2 ** attempt)

try:
    weather = fetch_daily_weather()
    print("Weather source:", WEATHER_URL)
    print("Weather timezone:", weather["timezone"].iloc[0])
    print("Weather records:", len(weather))
    display(weather.head())
except (requests.RequestException, ValueError) as error:
    weather = None
    print("Historical weather request failed:", error)
    print("No fabricated or fallback weather values were substituted.")

Weather source: https://archive-api.open-meteo.com/v1/archive
Weather timezone: America/New_York
Weather records: 30


,date,precipitation_sum,wind_speed_10m_max,pressure_msl_mean,source,timezone
0,2024-09-01,3.0,18.8,1014.1,Open-Meteo historical archive,America/New_York
1,2024-09-02,12.4,13.5,1014.1,Open-Meteo historical archive,America/New_York
2,2024-09-03,24.8,22.8,1015.6,Open-Meteo historical archive,America/New_York
3,2024-09-04,22.1,15.5,1017.7,Open-Meteo historical archive,America/New_York
4,2024-09-05,22.7,15.2,1017.3,Open-Meteo historical archive,America/New_York


In [28]:
sales_daily = (
    analysis_sales.groupby("date", as_index=False)
    .agg(transaction_count=("quantity", "size"),
         gross_units=("gross_units", "sum"),
         returned_units=("returned_units", "sum"),
         net_units=("net_units", "sum"))
)

if weather is not None:
    sales_weather = sales_daily.merge(
        weather, on="date", how="left", validate="one_to_one", indicator=True
    )
    assert len(sales_weather) == len(sales_daily), "Weather join changed the number of sales days"
    unmatched_dates = sales_weather.loc[sales_weather["_merge"] != "both", "date"]
    print("Unmatched sales dates:", unmatched_dates.dt.strftime("%Y-%m-%d").tolist())
    sales_weather = sales_weather.drop(columns="_merge")
    display(sales_weather.head())
else:
    sales_weather = None
    print("Weather join skipped until the real API request succeeds.")

Unmatched sales dates: []


,date,transaction_count,gross_units,returned_units,net_units,precipitation_sum,wind_speed_10m_max,pressure_msl_mean,source,timezone
0,2024-09-01,587,1132,4,1128,3.0,18.8,1014.1,Open-Meteo historical archive,America/New_York
1,2024-09-02,580,1084,2,1082,12.4,13.5,1014.1,Open-Meteo historical archive,America/New_York
2,2024-09-03,586,1123,4,1119,24.8,22.8,1015.6,Open-Meteo historical archive,America/New_York
3,2024-09-04,566,1017,8,1009,22.1,15.5,1017.7,Open-Meteo historical archive,America/New_York
4,2024-09-05,809,1560,6,1554,22.7,15.2,1017.3,Open-Meteo historical archive,America/New_York
